# 02. Data Preprocessing & Encoding Pipeline

## Mục tiêu
1. **Chia train/test theo thời gian**: Sparkov dùng đúng cách chia gốc của dataset — `fraudTrain.csv`
   (2019-01 → 2020-06) làm train, `fraudTest.csv` (2020-06 → 2020-12) làm test. ULB: sắp theo `Time`,
   20% cuối làm test. Train là quá khứ, test là tương lai — như một hệ thống phát hiện gian lận thật.
2. **Feature**: bỏ mã/ID và **mọi cột định danh khách hàng** (`config.CUSTOMER_IDENTITY_COLS`: tọa độ nhà,
   city, state, job, city_pop, tọa độ cửa hàng); thêm `hour`, `day_of_week`, `age` (`src/data/preprocess.py`).
3. **Encoding (spec mục 3)**: one-hot `gender`, `category`; Stratified K-fold Target Encoding `merchant`.
   Fit chỉ trên train (out-of-fold), test dùng mapping từ toàn bộ train.
4. **Lưu dữ liệu** vào `data/processed/`, giữ thứ tự thời gian (tune chia fold validation theo thứ tự dòng).

In [1]:
# 1. Setup & Imports
import sys
from pathlib import Path

# Đảm bảo import được module từ src/
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import joblib

from src.config import (
    RAW_DATA_DIR, PROCESSED_DATA_DIR, TRAIN_FILE, TEST_FILE,
    SEED, TARGET_COL, ONEHOT_COLS, TARGET_ENCODE_COLS, CUSTOMER_IDENTITY_COLS,
)
from src.data.encoding import encode_train, encode_test
from src.data.preprocess import preprocess_sparkov, split_ulb_by_time

print(f'SEED: {SEED}')
print(f'One-hot columns: {ONEHOT_COLS}')
print(f'Target encode columns: {TARGET_ENCODE_COLS}')
print(f'Cột định danh khách hàng bị bỏ: {CUSTOMER_IDENTITY_COLS}')

SEED: 42
One-hot columns: ['gender', 'category']
Target encode columns: ['merchant']
Cột định danh khách hàng bị bỏ: ['lat', 'long', 'city', 'state', 'job', 'city_pop', 'merch_lat', 'merch_long']


---
## 2. Load dữ liệu — train/test theo thời gian
`fraudTrain.csv` và `fraudTest.csv` là cách chia gốc của Sparkov, nối tiếp nhau theo thời gian.

In [2]:
# 2. Load raw files (giữ riêng train/test — KHÔNG gộp rồi chia ngẫu nhiên)
df_raw_train = pd.read_csv(RAW_DATA_DIR / TRAIN_FILE)
df_raw_test = pd.read_csv(RAW_DATA_DIR / TEST_FILE)

t_train = pd.to_datetime(df_raw_train['trans_date_trans_time'])
t_test = pd.to_datetime(df_raw_test['trans_date_trans_time'])
print(f'Train: {len(df_raw_train):,} dòng, {t_train.min()} → {t_train.max()}')
print(f'Test:  {len(df_raw_test):,} dòng, {t_test.min()} → {t_test.max()}')
assert t_train.max() <= t_test.min(), "Train phải kết thúc trước khi test bắt đầu"

Train: 1,296,675 dòng, 2019-01-01 00:00:18 → 2020-06-21 12:13:37
Test:  555,719 dòng, 2020-06-21 12:14:25 → 2020-12-31 23:59:34


In [3]:
# 3. Feature engineering + bỏ ID/định danh khách hàng (src/data/preprocess.py)
train_df = preprocess_sparkov(df_raw_train)
test_df = preprocess_sparkov(df_raw_test)

assert not set(CUSTOMER_IDENTITY_COLS) & set(train_df.columns)
print('Feature còn lại:', [c for c in train_df.columns if c != TARGET_COL])

Feature còn lại: ['merchant', 'category', 'amt', 'gender', 'hour', 'day_of_week', 'age']


---
## 3. Kiểm tra phép chia
Tỷ lệ fraud khác nhau giữa 2 giai đoạn là đặc điểm thật của dữ liệu (không phân tầng lại).

In [4]:
# 4. Kích thước + tỷ lệ fraud mỗi phía
print(f'Train set: {len(train_df):,} dòng, Fraud: {train_df[TARGET_COL].sum():,} ({train_df[TARGET_COL].mean():.4%})')
print(f'Test set:  {len(test_df):,} dòng, Fraud: {test_df[TARGET_COL].sum():,} ({test_df[TARGET_COL].mean():.4%})')

Train set: 1,296,675 dòng, Fraud: 7,506 (0.5789%)
Test set:  555,719 dòng, Fraud: 2,145 (0.3860%)


--- 
## 4. Encoding: One-Hot & Stratified K-Fold Target Encoding
- Fit chỉ trên `train_df` bằng `encode_train()`.
- Transform `test_df` bằng `encode_test()` sử dụng `encoding_maps` từ train.

In [5]:
# 5. Fit & Transform Encoding
print('Đang thực hiện Encoding trên tập train (Stratified K-fold)...')
encoded_train, encoding_maps = encode_train(
    train_df,
    target_col=TARGET_COL,
    onehot_cols=ONEHOT_COLS,
    target_encode_cols=TARGET_ENCODE_COLS,
    n_splits=5,
    smoothing=10,
    random_state=SEED,
)

print('Đang transform tập test với mapping từ train...')
encoded_test = encode_test(
    test_df,
    encoding_maps,
    target_encode_cols=TARGET_ENCODE_COLS,
)

print('Hoàn thành encoding!')
print(f'Shape train: {encoded_train.shape}')
print(f'Shape test:  {encoded_test.shape}')
assert list(encoded_train.columns) == list(encoded_test.columns), 'Cột train và test không khớp!'

Đang thực hiện Encoding trên tập train (Stratified K-fold)...


Đang transform tập test với mapping từ train...
Hoàn thành encoding!
Shape train: (1296675, 22)
Shape test:  (555719, 22)


--- 
## 5. Lưu dữ liệu đã tiền xử lý
Lưu kết quả ra `data/processed/` dưới dạng `.parquet` và `.csv` để phục vụ modeling.

In [6]:
# 6. Save processed datasets
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

train_out_parquet = PROCESSED_DATA_DIR / 'train_encoded.parquet'
test_out_parquet = PROCESSED_DATA_DIR / 'test_encoded.parquet'
maps_out = PROCESSED_DATA_DIR / 'encoding_maps.joblib'

encoded_train.to_parquet(train_out_parquet, index=False)
encoded_test.to_parquet(test_out_parquet, index=False)
joblib.dump(encoding_maps, maps_out)

print(f'✅ Đã lưu train (encoded): {train_out_parquet}')
print(f'✅ Đã lưu test (encoded):  {test_out_parquet}')
print(f'✅ Đã lưu maps:  {maps_out}')

# Lưu thêm bản RAW (chưa encode, chỉ mới feature engineering).
# Bắt buộc cho notebook 04 (CIES) — spec mục 7.1 yêu cầu encoding phải
# fit lại từ đầu trên mỗi bootstrap resample, KHÔNG được tái sử dụng
# encoding đã tính sẵn ở đây.
train_raw_parquet = PROCESSED_DATA_DIR / 'train_raw.parquet'
test_raw_parquet = PROCESSED_DATA_DIR / 'test_raw.parquet'

train_df.to_parquet(train_raw_parquet, index=False)
test_df.to_parquet(test_raw_parquet, index=False)

print(f'✅ Đã lưu train (raw, chưa encode): {train_raw_parquet}')
print(f'✅ Đã lưu test (raw, chưa encode):  {test_raw_parquet}')

✅ Đã lưu train (encoded): /Users/thetrung/Projects/Fraud Detection - CIES/data/processed/train_encoded.parquet
✅ Đã lưu test (encoded):  /Users/thetrung/Projects/Fraud Detection - CIES/data/processed/test_encoded.parquet
✅ Đã lưu maps:  /Users/thetrung/Projects/Fraud Detection - CIES/data/processed/encoding_maps.joblib


✅ Đã lưu train (raw, chưa encode): /Users/thetrung/Projects/Fraud Detection - CIES/data/processed/train_raw.parquet
✅ Đã lưu test (raw, chưa encode):  /Users/thetrung/Projects/Fraud Detection - CIES/data/processed/test_raw.parquet


--- 
## 6. Dataset Phụ (ULB Credit Card Fraud)
Tải và tiền xử lý dataset phụ ULB qua `kagglehub`.

In [7]:
# 7. Tải và xử lý dataset ULB
import kagglehub
from src.config import KAGGLE_DATASET_ULB, ULB_FILE

try:
    print(f'Đang tải dataset {KAGGLE_DATASET_ULB} từ Kaggle...')
    ulb_path = kagglehub.dataset_download(KAGGLE_DATASET_ULB)
    ulb_file = Path(ulb_path) / ULB_FILE
    df_ulb = pd.read_csv(ulb_file)
    print(f'ULB Dataset loaded: {df_ulb.shape[0]:,} dòng × {df_ulb.shape[1]} cột')
    print(f'Tỷ lệ fraud: {df_ulb["Class"].mean():.4%}')
    
    # Chia theo thời gian: 20% giao dịch cuối (theo Time) làm test
    ulb_train, ulb_test = split_ulb_by_time(df_ulb, test_size=0.20)
    print(f'ULB train: Time ≤ {ulb_train.Time.max():.0f}s, fraud {ulb_train.Class.mean():.4%} | '
          f'test: Time ≥ {ulb_test.Time.min():.0f}s, fraud {ulb_test.Class.mean():.4%}')
    ulb_train.to_parquet(PROCESSED_DATA_DIR / 'ulb_train.parquet', index=False)
    ulb_test.to_parquet(PROCESSED_DATA_DIR / 'ulb_test.parquet', index=False)
    print('✅ Đã lưu processed ULB dataset thành công!')
except Exception as e:
    print(f'Lưu ý về ULB dataset: {e}')
    print('Có thể tải thủ công hoặc chạy trên Kaggle/Colab có kết nối Kaggle API.')

Đang tải dataset mlg-ulb/creditcardfraud từ Kaggle...


ULB Dataset loaded: 284,807 dòng × 31 cột
Tỷ lệ fraud: 0.1727%
ULB train: Time ≤ 145248s, fraud 0.1830% | test: Time ≥ 145249s, fraud 0.1317%


✅ Đã lưu processed ULB dataset thành công!
